# Study 907 — Senior Loans vs High-Yield 🏦

**Senior secured loans sit *above* high-yield bonds in the capital stack — do you get paid a
"seniority premium" for it?**

Senior loans (BKLN, SRLN) are first-lien, better-recovery, floating-rate — and yield about
the same as high-yield bonds (HYG, JNK). The pitch: *same carry, less risk, a free seniority
premium.* We race the two sleeves, every Sharpe **excess of cash** (BIL), on the common
window 2011-03-03 → 2026-06-30 (bounded by BKLN's 2011 inception, so HY doesn't get the 2008
GFC the loans never saw).

*Numbers below are the frozen headline (`docs/results.md`, Fingerprint `e09ddb919d86`); the live
cell runs the fast synthetic control. Short-history caveat: SRLN only lists 2013 — named on
the Signal axis.*


## 1. What 'senior secured' actually buys

In a default, first-lien **loans** get paid before **bonds** — historical recoveries ~60–80% for senior secured loans vs ~35–45% for senior unsecured bonds. And loans **float** (coupon resets with short rates), so they carry almost no interest-rate duration. Two real, valuable features. The question is whether they add up to a *higher risk-adjusted return* — or just a *calmer* one.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../../..'))
import numpy as np
from loans_vs_hy import data, strategy as st
R = {'start': '2011-03-03', 'end': '2026-06-30', 'n_days': 3854, 'fp': 'e09ddb919d86', 'bkln': (3.71, 5.8, 0.414, -24.2), 'srln': (3.79, 5.4, 0.411, -22.3), 'hyg': (4.72, 8.2, 0.431, -22.0), 'jnk': (4.61, 8.1, 0.421, -22.9), 'loans': (3.82, 5.4, 0.457, -23.2), 'hy': (4.67, 8.1, 0.428, -22.5), 'ief': (2.38, 6.5, 0.177, -23.9), 'flag_adv': -0.017, 'flag_spread_pct': -1.14, 'flag_spread_t': -0.95, 'comp_adv': 0.029, 'comp_spread_pct': -1.0, 'comp_spread_t': -0.83, 'boot_adv': 0.029, 'boot_lo': -0.258, 'boot_hi': 0.47, 'boot_win': 62, 'eras': [('2011-15 energy build-up', 0.54, 0.43, 0.1, -0.49), ('2016-19', 1.64, 1.16, 0.48, -1.38), ('2020-22 COVID + hike', 0.05, -0.09, 0.13, 0.49), ('2023-26', 0.96, 0.69, 0.27, -0.44)], 'stress': [('Energy wave 2015-16', -6.8, -5.3, -12.1, -15.0), ('COVID crash 2020', -23.8, -22.3, -21.9, -22.8), ('2022 rate shock', -4.5, -7.4, -14.6, -15.8)], 'cost5_net': -2.82, 'cost5_t': -2.34, 'cost3_net': -2.14, 'cost3_t': -1.77, 'gross_ls': -1.02, 'null_adv': -0.061, 'null_sd': 0.253, 'planted_adv': 0.334, 'planted_win': 93}
print('LOANS sleeve : CAGR %.2f%%  vol %.1f%%  excess-Sharpe %+.3f  maxDD %.1f%%'
      % R['loans'])
print('HY    sleeve : CAGR %.2f%%  vol %.1f%%  excess-Sharpe %+.3f  maxDD %.1f%%'
      % R['hy'])
print()
print('Loans are a THIRD less volatile (%.1f%% vs %.1f%%) ...' % (R['loans'][1], R['hy'][1]))
print('... but earn a full point LESS per year (%.2f%% vs %.2f%% CAGR).' % (R['loans'][0], R['hy'][0]))
print('The two forces nearly cancel: Sharpe %+.3f vs %+.3f.' % (R['loans'][2], R['hy'][2]))

LOANS sleeve : CAGR 3.82%  vol 5.4%  excess-Sharpe +0.457  maxDD -23.2%
HY    sleeve : CAGR 4.67%  vol 8.1%  excess-Sharpe +0.428  maxDD -22.5%

Loans are a THIRD less volatile (5.4% vs 8.1%) ...
... but earn a full point LESS per year (3.82% vs 4.67% CAGR).
The two forces nearly cancel: Sharpe +0.457 vs +0.428.


## 2. Lower vol is not a free premium

The seniority discount buys **calm, not extra carry**. On a risk-adjusted basis the loan sleeve's advantage is a rounding error — and its *sign flips* depending on whether you use the single flagship pair or the two-ETF composite.

In [2]:
R = {'start': '2011-03-03', 'end': '2026-06-30', 'n_days': 3854, 'fp': 'e09ddb919d86', 'bkln': (3.71, 5.8, 0.414, -24.2), 'srln': (3.79, 5.4, 0.411, -22.3), 'hyg': (4.72, 8.2, 0.431, -22.0), 'jnk': (4.61, 8.1, 0.421, -22.9), 'loans': (3.82, 5.4, 0.457, -23.2), 'hy': (4.67, 8.1, 0.428, -22.5), 'ief': (2.38, 6.5, 0.177, -23.9), 'flag_adv': -0.017, 'flag_spread_pct': -1.14, 'flag_spread_t': -0.95, 'comp_adv': 0.029, 'comp_spread_pct': -1.0, 'comp_spread_t': -0.83, 'boot_adv': 0.029, 'boot_lo': -0.258, 'boot_hi': 0.47, 'boot_win': 62, 'eras': [('2011-15 energy build-up', 0.54, 0.43, 0.1, -0.49), ('2016-19', 1.64, 1.16, 0.48, -1.38), ('2020-22 COVID + hike', 0.05, -0.09, 0.13, 0.49), ('2023-26', 0.96, 0.69, 0.27, -0.44)], 'stress': [('Energy wave 2015-16', -6.8, -5.3, -12.1, -15.0), ('COVID crash 2020', -23.8, -22.3, -21.9, -22.8), ('2022 rate shock', -4.5, -7.4, -14.6, -15.8)], 'cost5_net': -2.82, 'cost5_t': -2.34, 'cost3_net': -2.14, 'cost3_t': -1.77, 'gross_ls': -1.02, 'null_adv': -0.061, 'null_sd': 0.253, 'planted_adv': 0.334, 'planted_win': 93}
print('Flagship BKLN vs HYG : Sharpe advantage %+.3f  (return spread %.2f%%/yr, t=%.2f)'
      % (R['flag_adv'], R['flag_spread_pct'], R['flag_spread_t']))
print('Composite  L vs HY   : Sharpe advantage %+.3f  (return spread %.2f%%/yr, t=%.2f)'
      % (R['comp_adv'], R['comp_spread_pct'], R['comp_spread_t']))
print()
print('Bootstrap on the advantage: %+.3f, 95%% CI [%+.3f, %+.3f], loans win %d%% of draws'
      % (R['boot_adv'], R['boot_lo'], R['boot_hi'], R['boot_win']))
print('-> the interval straddles zero: risk-adjusted, loans and HY are a WASH.')

Flagship BKLN vs HYG : Sharpe advantage -0.017  (return spread -1.14%/yr, t=-0.95)
Composite  L vs HY   : Sharpe advantage +0.029  (return spread -1.00%/yr, t=-0.83)

Bootstrap on the advantage: +0.029, 95% CI [-0.258, +0.470], loans win 62% of draws
-> the interval straddles zero: risk-adjusted, loans and HY are a WASH.


## 3. Where seniority helps — and where it bites

The honest split. When the pain is **spreads or rates**, seniority + the floating coupon deliver: loans lose about **half** what HY loses. But in a **pure liquidity crisis** the loan sleeve — the *less liquid* leg, sold at forced-seller discounts — gaps as hard or **harder** than HY, exactly when you wanted protection.

In [3]:
R = {'start': '2011-03-03', 'end': '2026-06-30', 'n_days': 3854, 'fp': 'e09ddb919d86', 'bkln': (3.71, 5.8, 0.414, -24.2), 'srln': (3.79, 5.4, 0.411, -22.3), 'hyg': (4.72, 8.2, 0.431, -22.0), 'jnk': (4.61, 8.1, 0.421, -22.9), 'loans': (3.82, 5.4, 0.457, -23.2), 'hy': (4.67, 8.1, 0.428, -22.5), 'ief': (2.38, 6.5, 0.177, -23.9), 'flag_adv': -0.017, 'flag_spread_pct': -1.14, 'flag_spread_t': -0.95, 'comp_adv': 0.029, 'comp_spread_pct': -1.0, 'comp_spread_t': -0.83, 'boot_adv': 0.029, 'boot_lo': -0.258, 'boot_hi': 0.47, 'boot_win': 62, 'eras': [('2011-15 energy build-up', 0.54, 0.43, 0.1, -0.49), ('2016-19', 1.64, 1.16, 0.48, -1.38), ('2020-22 COVID + hike', 0.05, -0.09, 0.13, 0.49), ('2023-26', 0.96, 0.69, 0.27, -0.44)], 'stress': [('Energy wave 2015-16', -6.8, -5.3, -12.1, -15.0), ('COVID crash 2020', -23.8, -22.3, -21.9, -22.8), ('2022 rate shock', -4.5, -7.4, -14.6, -15.8)], 'cost5_net': -2.82, 'cost5_t': -2.34, 'cost3_net': -2.14, 'cost3_t': -1.77, 'gross_ls': -1.02, 'null_adv': -0.061, 'null_sd': 0.253, 'planted_adv': 0.334, 'planted_win': 93}
print('%-22s %7s %7s %7s %7s' % ('episode','BKLN','SRLN','HYG','JNK'))
for e in R['stress']:
    print('%-22s %6.1f%% %6.1f%% %6.1f%% %6.1f%%' % e)
print()
print('Energy & 2022: loans lose ~half of HY (seniority + floating rate work).')
print('COVID: BKLN -23.8%% vs HYG -21.9%% -- loans gap WORSE (liquidity run).')

episode                   BKLN    SRLN     HYG     JNK
Energy wave 2015-16      -6.8%   -5.3%  -12.1%  -15.0%
COVID crash 2020        -23.8%  -22.3%  -21.9%  -22.8%
2022 rate shock          -4.5%   -7.4%  -14.6%  -15.8%

Energy & 2022: loans lose ~half of HY (seniority + floating rate work).
COVID: BKLN -23.8%% vs HYG -21.9%% -- loans gap WORSE (liquidity run).


## 4. The trade loses money

To actually *harvest* seniority you'd go **long loans / short HY**. But loans earn **less**, so the spread is negative before you pay a cent — and after borrow on the short HY leg it bleeds 2–3%/yr.

In [4]:
R = {'start': '2011-03-03', 'end': '2026-06-30', 'n_days': 3854, 'fp': 'e09ddb919d86', 'bkln': (3.71, 5.8, 0.414, -24.2), 'srln': (3.79, 5.4, 0.411, -22.3), 'hyg': (4.72, 8.2, 0.431, -22.0), 'jnk': (4.61, 8.1, 0.421, -22.9), 'loans': (3.82, 5.4, 0.457, -23.2), 'hy': (4.67, 8.1, 0.428, -22.5), 'ief': (2.38, 6.5, 0.177, -23.9), 'flag_adv': -0.017, 'flag_spread_pct': -1.14, 'flag_spread_t': -0.95, 'comp_adv': 0.029, 'comp_spread_pct': -1.0, 'comp_spread_t': -0.83, 'boot_adv': 0.029, 'boot_lo': -0.258, 'boot_hi': 0.47, 'boot_win': 62, 'eras': [('2011-15 energy build-up', 0.54, 0.43, 0.1, -0.49), ('2016-19', 1.64, 1.16, 0.48, -1.38), ('2020-22 COVID + hike', 0.05, -0.09, 0.13, 0.49), ('2023-26', 0.96, 0.69, 0.27, -0.44)], 'stress': [('Energy wave 2015-16', -6.8, -5.3, -12.1, -15.0), ('COVID crash 2020', -23.8, -22.3, -21.9, -22.8), ('2022 rate shock', -4.5, -7.4, -14.6, -15.8)], 'cost5_net': -2.82, 'cost5_t': -2.34, 'cost3_net': -2.14, 'cost3_t': -1.77, 'gross_ls': -1.02, 'null_adv': -0.061, 'null_sd': 0.253, 'planted_adv': 0.334, 'planted_win': 93}
print('long loans / short HY, dollar-neutral:')
print('  gross spread            : %+.2f%%/yr' % R['gross_ls'])
print('  net (5bps + 60bps borrow): %+.2f%%/yr  (t=%.2f)' % (R['cost5_net'], R['cost5_t']))
print('  net (3bps + 40bps borrow): %+.2f%%/yr  (t=%.2f)' % (R['cost3_net'], R['cost3_t']))

long loans / short HY, dollar-neutral:
  gross spread            : -1.02%/yr
  net (5bps + 60bps borrow): -2.82%/yr  (t=-2.34)
  net (3bps + 40bps borrow): -2.14%/yr  (t=-1.77)


## 5. The takeaway

- **Signal — WEAK.** Lower vol is real; a higher *risk-adjusted return* is not — the advantage's sign flips with construction and its bootstrap CI straddles zero.
- **Tradability — MIRAGE.** The natural trade is negative gross and −2 to −3%/yr after costs.
- **Free premium? — BUSTED.** Seniority halves your loss in rate/spread selloffs but costs total return and gaps worse in a liquidity run.

Senior loans are the **calmer** cousin of high-yield, not the **richer** one.